# STAT 8330 Old Final Exam Solutions: Fall 2023

This notebook gives complete Python solutions for the Fall 2023 old final exam. It is self-contained, exam-oriented, and written with comments on each executable line.

Source PDF in this folder: `FInalExam8330_Fall2023.pdf`.

## Setup

The 2023 exam focuses on simulation-based model assessment:

- lasso prediction error as the penalty changes,
- LOOCV tuning for a kernel smoother,
- simulation for the distribution of a student's last final-exam day.

Documentation links:

- [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)

In [ ]:
import numpy as np  # Import numerical arrays and random-number generation tools.
import pandas as pd  # Import DataFrame and Series tools for tabular summaries.
import matplotlib.pyplot as plt  # Import plotting tools for simulation figures.
from IPython.display import display  # Import display so DataFrames render cleanly in notebooks.
from sklearn.linear_model import Lasso  # Import lasso regression estimator.
from sklearn.metrics import mean_squared_error  # Import prediction-error metric.
from sklearn.pipeline import make_pipeline  # Import pipeline helper for scaling plus lasso.
from sklearn.preprocessing import StandardScaler  # Import standardization for fair lasso penalties.
RANDOM_SEED = 123  # Store the random seed in one visible place.
rng = np.random.default_rng(RANDOM_SEED)  # Create a reproducible random-number generator.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness used by some libraries.
plt.style.use('default')  # Use a predictable plotting style.


## Problem 1: Lasso Penalty Simulation Study

**Prompt summary.** Devise scenarios where prediction error favors different lasso penalty values. In each simulation replicate, fit the model on training data and compute prediction error on test data. Plot mean prediction error versus penalty.

**Theory.** The lasso minimizes squared error plus an L1 penalty. Small `alpha` values behave more like least squares. Large `alpha` values shrink coefficients aggressively and can set many to zero. Larger penalties tend to help when many predictors are irrelevant or noise is high; smaller penalties tend to help when many predictors carry real signal.

In [ ]:
def make_lasso_data(n, p, beta, noise, random_state):  # Define a simulator for linear-regression data with a chosen coefficient vector.
    rng = np.random.default_rng(random_state)  # Create a reproducible random-number generator.
    X = rng.normal(0.0, 1.0, size=(n, p))  # Simulate standardized predictors.
    y = X @ beta + rng.normal(0.0, noise, size=n)  # Generate the response from signal plus Gaussian noise.
    return X, y  # Return predictors and response.

def scenario_beta(p, scenario_name):  # Define true coefficient vectors for different lasso scenarios.
    beta = np.zeros(p)  # Start with all predictors irrelevant.
    if scenario_name == 'sparse strong signal':  # Check for a scenario with a few strong predictors.
        beta[:4] = np.array([3.0, -2.5, 2.0, -1.5])  # Set four large nonzero coefficients.
    elif scenario_name == 'dense weak signal':  # Check for a scenario with many small real effects.
        beta[:14] = np.linspace(0.9, 0.25, 14)  # Set many modest nonzero coefficients.
    elif scenario_name == 'sparse noisy signal':  # Check for a sparse scenario with high observation noise.
        beta[:4] = np.array([2.0, -1.5, 1.0, -0.8])  # Set a weaker sparse signal.
    else:  # Handle unexpected scenario names.
        raise ValueError('Unknown scenario_name')  # Stop with a clear error message.
    return beta  # Return the scenario-specific coefficient vector.

def lasso_penalty_simulation(alpha_grid, scenarios, nrep=30, n_train=80, n_test=400, p=25, random_state=123):  # Define the requested lasso simulation study.
    rows = []  # Create a list for replicate-level prediction errors.
    for scenario_name, noise in scenarios:  # Loop over scenario names and their noise levels.
        beta = scenario_beta(p, scenario_name)  # Get the true coefficients for this scenario.
        for rep in range(nrep):  # Repeat the train/test experiment many times.
            X_train, y_train = make_lasso_data(n_train, p, beta, noise, random_state + 1000 * rep + 17)  # Simulate training data.
            X_test, y_test = make_lasso_data(n_test, p, beta, noise, random_state + 1000 * rep + 29)  # Simulate independent test data.
            for alpha in alpha_grid:  # Loop over candidate lasso penalty values.
                model = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=30000))  # Build standardized lasso for this alpha.
                model.fit(X_train, y_train)  # Fit lasso on the training data only.
                test_pred = model.predict(X_test)  # Predict independent test responses.
                test_mse = mean_squared_error(y_test, test_pred)  # Compute test mean squared error.
                rows.append({'scenario': scenario_name, 'noise': noise, 'rep': rep, 'alpha': alpha, 'test_mse': test_mse})  # Store this replicate result.
    results = pd.DataFrame(rows)  # Convert all simulation records to a DataFrame.
    return results  # Return replicate-level simulation results.


In [ ]:
alpha_grid = np.logspace(-3, 0.7, 14)  # Define candidate lasso penalties on a log scale.
scenarios = [('sparse strong signal', 1.0), ('dense weak signal', 1.0), ('sparse noisy signal', 3.0)]  # Define scenarios expected to favor different penalties.
lasso_results = lasso_penalty_simulation(alpha_grid, scenarios, nrep=25, random_state=RANDOM_SEED)  # Run the simulation study.
mean_curve = lasso_results.groupby(['scenario', 'alpha'])['test_mse'].mean().reset_index()  # Average test MSE across replicates.
best_alpha_table = mean_curve.loc[mean_curve.groupby('scenario')['test_mse'].idxmin()].reset_index(drop=True)  # Find the best alpha for each scenario.
display(best_alpha_table)  # Display the scenario-specific best penalties.
plt.figure(figsize=(7, 4))  # Create a plotting canvas for penalty curves.
for scenario_name in mean_curve['scenario'].unique():  # Loop over scenarios for plotting.
    subset = mean_curve[mean_curve['scenario'] == scenario_name]  # Select one scenario's mean curve.
    plt.plot(subset['alpha'], subset['test_mse'], marker='o', label=scenario_name)  # Plot mean test MSE versus alpha.
plt.xscale('log')  # Use a log scale because penalties span orders of magnitude.
plt.xlabel('lasso penalty alpha')  # Label the tuning axis.
plt.ylabel('mean test MSE')  # Label the prediction-error axis.
plt.title('Lasso prediction error depends on the penalty and scenario')  # Title the simulation plot.
plt.legend()  # Show scenario labels.
plt.show()  # Render the lasso simulation plot.


**Problem 1 findings.** In a sparse, noisy scenario, some shrinkage is helpful because many estimated coefficients are noise. In a dense weak-signal scenario, overly large penalties can hurt because the lasso shrinks many real but modest effects. The correct penalty is therefore scenario-dependent and should be judged by test error or cross-validation.

## Problem 2: LOOCV Tuning For A Kernel Smoother

**Prompt summary.** Write a function that uses leave-one-out cross-validation to choose the smoothing parameter for a kernel smoother. The inputs should be a data frame, the name of a kernel smoothing function, and candidate smoothing values. Test it with a Nadaraya-Watson function.

**Theory.** A kernel smoother predicts by a distance-weighted average of nearby responses. The bandwidth controls locality. LOOCV estimates prediction error by leaving out one point, predicting it from all other points, and averaging squared errors.

In [ ]:
def make_kernel_demo_data(n=90, noise=0.35, random_state=123):  # Define a simulator for one-dimensional nonlinear regression.
    rng = np.random.default_rng(random_state)  # Create a reproducible random-number generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Simulate sorted predictor values for clean plots.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true smooth mean function.
    y = signal + rng.normal(0.0, noise, size=n)  # Add Gaussian observation noise.
    dat = pd.DataFrame({'x': x, 'y': y})  # Store the data in a two-column DataFrame.
    return dat  # Return the simulated data frame.

def NadWat(dat, x_new, h):  # Define the Nadaraya-Watson Gaussian kernel smoother used for testing.
    x_train = dat.iloc[:, 0].to_numpy()  # Extract training predictor values.
    y_train = dat.iloc[:, 1].to_numpy()  # Extract training response values.
    x_new = np.asarray(x_new)  # Convert prediction locations to an array.
    predictions = []  # Create a list for predictions.
    for x0 in x_new:  # Predict one new location at a time.
        weights = np.exp(-0.5 * ((x_train - x0) / h) ** 2)  # Compute Gaussian kernel weights around x0.
        prediction = np.sum(weights * y_train) / np.sum(weights)  # Compute the weighted average response.
        predictions.append(prediction)  # Store the prediction.
    return np.array(predictions)  # Return predictions as a NumPy array.

def find_kernel_smoothing_parameter_loocv(dat, smoother_function_name, smooth_values):  # Define the LOOCV bandwidth-selection function.
    if isinstance(smoother_function_name, str):  # Check whether the smoother was passed by name.
        smoother_function = globals()[smoother_function_name]  # Look up the named smoother function in the notebook namespace.
    else:  # Handle the case where the smoother itself was passed.
        smoother_function = smoother_function_name  # Use the callable directly.
    x_all = dat.iloc[:, 0].to_numpy()  # Extract all predictor values.
    y_all = dat.iloc[:, 1].to_numpy()  # Extract all response values.
    rows = []  # Create a list for candidate bandwidth results.
    for smooth_value in smooth_values:  # Loop over smoothing parameter candidates.
        squared_errors = []  # Create a list for leave-one-out squared errors.
        for i in range(len(dat)):  # Leave out each observation once.
            train_dat = dat.drop(dat.index[i]).reset_index(drop=True)  # Remove observation i from the training data.
            prediction = smoother_function(train_dat, [x_all[i]], smooth_value)[0]  # Predict the left-out response using all other observations.
            squared_error = (y_all[i] - prediction) ** 2  # Compute the left-out squared prediction error.
            squared_errors.append(squared_error)  # Store the squared error.
        rows.append({'smooth_value': smooth_value, 'loocv_mse': np.mean(squared_errors)})  # Store average LOOCV MSE for this candidate.
    cv_table = pd.DataFrame(rows)  # Convert results to a DataFrame.
    best_value = float(cv_table.loc[cv_table['loocv_mse'].idxmin(), 'smooth_value'])  # Select the candidate with smallest LOOCV MSE.
    find_kernel_smoothing_parameter_loocv.last_cv_table = cv_table  # Save the CV table for inspection after the call.
    return best_value  # Return the optimal smoothing parameter value.


In [ ]:
kernel_dat = make_kernel_demo_data(n=90, noise=0.35, random_state=RANDOM_SEED)  # Simulate data for testing the smoother function.
bandwidth_grid = np.linspace(0.05, 0.80, 12)  # Define candidate bandwidth values.
best_bandwidth = find_kernel_smoothing_parameter_loocv(kernel_dat, 'NadWat', bandwidth_grid)  # Tune the Nadaraya-Watson bandwidth by LOOCV.
display(find_kernel_smoothing_parameter_loocv.last_cv_table)  # Display the LOOCV result table.
x_grid = np.linspace(kernel_dat['x'].min(), kernel_dat['x'].max(), 200)  # Create a smooth x-grid for plotting.
y_grid = NadWat(kernel_dat, x_grid, best_bandwidth)  # Predict the fitted curve using the selected bandwidth.
print(f'Best bandwidth selected by LOOCV: {best_bandwidth:.3f}')  # Print the selected smoothing value.
plt.figure(figsize=(6, 4))  # Create a plotting canvas.
plt.scatter(kernel_dat['x'], kernel_dat['y'], s=22, alpha=0.60, label='simulated data')  # Plot the observed data.
plt.plot(x_grid, y_grid, color='red', label=f'best h = {best_bandwidth:.2f}')  # Plot the selected kernel smoother.
plt.xlabel('x')  # Label the predictor axis.
plt.ylabel('y')  # Label the response axis.
plt.title('Nadaraya-Watson bandwidth selected by LOOCV')  # Title the plot.
plt.legend()  # Show plot labels.
plt.show()  # Render the plot.


**Problem 2 exam takeaway.** Bandwidth is the kernel smoother's complexity control. Small bandwidths can be noisy; large bandwidths can oversmooth. LOOCV directly estimates the left-out prediction error for each candidate.

## Problem 3: Simulating The Day Of A Student's Last Final Exam

**Prompt summary.** A school has several final-exam days and a fixed number of slots per day. A student has `nExams` exams, with no two exams in the same slot. Write `exam_day_distribution(Days, nSlots, nExams, nrep=1000)` to estimate the probability that the student's last exam occurs on each day.

**Important note.** The problem statement says exam slots are sampled without replacement because two exams cannot occur at the same time. The code below follows that condition.

In [ ]:
def exam_day_distribution(Days, nSlots, nExams, nrep=1000, random_state=123):  # Define the requested final-exam-day simulation function.
    rng = np.random.default_rng(random_state)  # Create a reproducible random-number generator.
    Days = list(Days)  # Convert the day input to a list so indexing is simple.
    total_slots = len(Days) * nSlots  # Compute the total number of possible exam slots.
    if nExams > total_slots:  # Check whether the requested number of exams is possible.
        raise ValueError('nExams cannot exceed the total number of exam slots')  # Stop if there are not enough unique slots.
    counts = pd.Series(0, index=Days, dtype=float)  # Create a counter for the last-exam day.
    slot_numbers = np.arange(total_slots)  # Index all slots from zero to total_slots minus one.
    for rep in range(nrep):  # Repeat the schedule simulation many times.
        sampled_slots = rng.choice(slot_numbers, size=nExams, replace=False)  # Sample unique exam slots without replacement.
        last_slot = int(np.max(sampled_slots))  # The latest exam is the largest slot index.
        last_day_index = last_slot // nSlots  # Convert the slot index to a day index using integer division.
        last_day = Days[last_day_index]  # Convert the day index to the corresponding day label.
        counts.loc[last_day] = counts.loc[last_day] + 1.0  # Add one count for the simulated last-exam day.
    probabilities = counts / nrep  # Convert counts to simulated probabilities.
    return probabilities  # Return the probability distribution over final-exam days.


In [ ]:
uga_days = ['Wednesday', 'Thursday', 'Friday', 'Monday', 'Tuesday']  # Define the five final-exam days from the prompt.
uga_distribution = exam_day_distribution(uga_days, nSlots=3, nExams=3, nrep=100000, random_state=RANDOM_SEED)  # Estimate the prompt's probability distribution.
small_distribution = exam_day_distribution(['Day 1', 'Day 2', 'Day 3'], nSlots=2, nExams=2, nrep=50000, random_state=RANDOM_SEED + 1)  # Run a second example with a smaller schedule.
print('UGA-style example with five days, three slots per day, and three exams:')  # Label the main example output.
display(uga_distribution)  # Display the simulated UGA-style distribution.
print('Second example with three days, two slots per day, and two exams:')  # Label the second example output.
display(small_distribution)  # Display the second simulated distribution.
plt.figure(figsize=(7, 4))  # Create a bar-plot canvas.
uga_distribution.plot(kind='bar')  # Draw a bar plot of the UGA-style probabilities.
plt.ylabel('simulated probability')  # Label the probability axis.
plt.title('Distribution of the day of the last final exam')  # Title the plot.
plt.xticks(rotation=25)  # Rotate labels for readability.
plt.show()  # Render the bar plot.


**Problem 3 exam takeaway.** The last final exam is the maximum sampled slot. After finding that maximum slot, integer division by the number of slots per day converts the slot to the day index.